# 05 · Mamba vs Transformer Mini —— 线性流水线 vs 二次全员会的同台

**家族位置**：`05_Transformer_NLP` 第 5 站（拓展对照）。01 把 MHA/因果/PE 手写满分，02/03/04 拼成 BERT/GPT/T5 三变体，本章用 **Mamba selective SSM**（线性扫）与 **GPT 因果 Transformer**（二次全注意力）在同 `循环计数` toy 上同参对照，并用理论与实测说明“同样学到 next 1.0，显存/算力随 S 增长谁线性、谁二次”。与 `04-04` 共用极简 Mamba 实现，呼应“04 的 RNN 复兴”视角。

**学习目标**
1. SSM 极简：`h_t = exp(A·dt)·h_{t-1}+B·x` selective（dt/B/C 随输入变）vs 注意力 `QKᵀ`
2. 同 LM 任务同预算，Transformer 次二次、Mamba 线性：S=8/16/32 下算力/显存趋势
3. 同台 next-acc：循环计数 toy 上双高分，说明线性也能学有序
4. 图：loss 同台 + seq/tok 柱状 + 计算量/显存随 S 曲线

## 1. 原理：从“全员同时开会”到“流水线选择性记忆”

### 通俗理解

**一句话**：Transformer 每步都看全句（S×S 矩阵，算力 S²·d），Mamba 每步只更新一次隐藏态再往下传（S·d·N，算力 S·d，显存 O(1) 递推）。

**比喻**：Transformer 像 8 人开会每人都要听 7 人发言（64 票）；Mamba 像流水线传送带——每人加工完传给下一人，手里只留一个 `h`（选择性 decide 记不记，靠 `dt/B/C` 随输入定）。

### 结构账

```
Transformer 因果： Attn = softmax(QKᵀ/√d + 下三角)·V   O(S²·d) 全矩阵
Mamba SSM：       h_t = exp(A·dt)·h_{t-1} + B·x_t ;  y_t = C·h_t + D·x_t ;  dt/B/C = Linear(x_t) selective
                一次前向线性扫 O(S·D·N)，无 S×S 矩阵
本实验：  GPT(d32/h4/L2) 17,344 参 vs Mamba(d32/L2/N=8) ~14k 参   同 LM 800/200 S=8，Adam 8e-3 40ep  CPU
```

- **评估**：next-token acc + loss 曲线；算力=理论 FLOPs 随 S，外加实测耗时小 benchmark

In [ ]:
import sys
from pathlib import Path
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import make_lm_data
from common.models import GPTForLM, MambaForLM, set_torch_seed
from common.utils import set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
print("torch:", torch.__version__)

VOCAB, SEQ_LEN = 8, 8
N_TRAIN, N_TEST = 800, 200
EPOCHS, BATCH, LR = 40, 32, 8e-3
X_all = make_lm_data(N_TRAIN+N_TEST, SEQ_LEN, VOCAB, seed=0)
X_train, X_test = X_all[:N_TRAIN], X_all[N_TRAIN:]
print(f"循环计数 S={SEQ_LEN} vocab={VOCAB} train {len(X_train)}/test {len(X_test)}")


## 2. 同台训练：GPT 因果 vs Mamba 线性（同一 LM，同一预算）

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def train_lm(model, Xtr, Xte, epochs=EPOCHS, batch=BATCH, lr=LR, seed=0):
    set_torch_seed(seed)
    xt = torch.tensor(Xtr, dtype=torch.long); xe = torch.tensor(Xte, dtype=torch.long)
    loader = DataLoader(TensorDataset(xt), batch_size=batch, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    h_loss, h_acc = [], []
    for ep in range(1, epochs+1):
        model.train()
        tot=0
        for (xb,) in loader:
            logits = model(xb)
            loss = lossf(logits[:, :-1].reshape(-1, VOCAB), xb[:, 1:].reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()*len(xb)
        h_loss.append(tot/len(Xtr))
        model.eval()
        with torch.no_grad():
            lg = model(xe); pred = lg[:, :-1].argmax(dim=-1)
            acc = (pred==xe[:,1:]).float().mean().item()
            h_acc.append(acc)
    model.eval()
    with torch.no_grad():
        lg = model(xe); pred = lg[:, :-1].argmax(dim=-1)
        acc = (pred==xe[:,1:]).float().mean().item()
    return h_loss, h_acc, acc, pred

gpt = GPTForLM(VOCAB, d_model=32, n_head=4, n_layer=2, d_ff=64, max_len=64)
mamba = MambaForLM(VOCAB, d_model=32, n_layer=2, d_state=8, max_len=64)
for name, model in [("GPT", gpt), ("Mamba", mamba)]:
    print(f"{name} params={sum(p.numel() for p in model.parameters() if p.requires_grad)}")

hl_gpt, ha_gpt, acc_gpt, pred_gpt = train_lm(gpt, X_train, X_test, seed=0)
hl_mamba, ha_mamba, acc_mamba, pred_mamba = train_lm(mamba, X_train, X_test, seed=0)
print(f"GPT next-acc={acc_gpt:.4f}")
print(f"Mamba next-acc={acc_mamba:.4f}")

# fig1：loss 双曲线
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(hl_gpt, label="GPT", color="#4C72B0")
ax.plot(hl_mamba, label="Mamba", color="#55A868")
ax.set_xlabel("epoch"); ax.set_ylabel("CE loss")
ax.set_title("GPT vs Mamba（同 LM 同预算，循环 S=8）")
ax.legend()
plt.tight_layout()
plt.savefig(FIGS / "fig1_loss.png", dpi=150, bbox_inches="tight")
plt.show()

# fig2：next-acc 柱状 vs 随机 1/8
fig, ax = plt.subplots(figsize=(4.8, 3.6))
ax.bar(["基线 1/8", "GPT", "Mamba"], [1/VOCAB, acc_gpt, acc_mamba], color=["#BDC3C7","#4C72B0","#55A868"])
ax.set_ylim(0,1.12); ax.set_ylabel("next-token acc")
for i, v in enumerate([1/VOCAB, acc_gpt, acc_mamba]):
    ax.text(i, v+0.03, f"{v:.3f}", ha="center")
ax.set_title("同一循环计数：二次 vs 线性同台")
plt.tight_layout()
plt.savefig(FIGS / "fig2_bar.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. 线性 vs 二次：随 S 增长的算力/显存趋势（理论 + 微 benchmark）

In [ ]:
# fig3：理论 FLOPs 随 S（GPT O(S^2 d) vs Mamba O(S d N)）
S_list = [8, 16, 32, 64, 128]
d, N = 32, 8
# 粗算：GPT 每层 ~ 4*S^2*d + 2*S*d^2；Mamba 每层 ~ S*d*N*4；取相对曲线归一到 S=8=1
flops_gpt = [(4*s*s*d + 2*s*d*d) for s in S_list]
flops_mamba = [(s*d*N*6) for s in S_list]
fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.plot(S_list, [f/flops_gpt[0] for f in flops_gpt], marker="o", label="GPT 二次 O(S^2)", color="#4C72B0")
ax.plot(S_list, [f/flops_mamba[0] for f in flops_mamba], marker="s", label="Mamba 线性 O(S)", color="#55A868")
ax.set_xlabel("序列长度 S"); ax.set_ylabel("相对算力（S=8 归一）")
ax.set_title("算力随 S 增长：线性流水线 vs 二次全员会（理论）")
ax.legend()
plt.tight_layout()
plt.savefig(FIGS / "fig3_flops.png", dpi=150, bbox_inches="tight")
plt.show()

# fig4：实测耗时微 bench（CPU，B=32，前向 50 次均值，S sweep）
def bench(model, S, B=32, it=50):
    model.eval()
    x = torch.randint(0, VOCAB, (B, S))
    # warmup
    with torch.no_grad():
        for _ in range(5): _ = model(x)
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(it): _ = model(x)
    return (time.perf_counter()-t0)/it*1000

bench_S = [8, 16, 32, 64]
t_gpt = [bench(gpt, s) for s in bench_S]
t_mamba = [bench(mamba, s) for s in bench_S]
fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.plot(bench_S, t_gpt, marker="o", label="GPT", color="#4C72B0")
ax.plot(bench_S, t_mamba, marker="s", label="Mamba", color="#55A868")
ax.set_xlabel("S"); ax.set_ylabel("前向耗时 ms (B=32, 均值)")
ax.set_title("实测耗时随 S（CPU toy，供趋势，非大模型绝对值）")
ax.legend()
plt.tight_layout()
plt.savefig(FIGS / "fig4_bench.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"bench ms GPT={t_gpt} Mamba={t_mamba}")


## 4. 生成抽样对比

同一前缀 3→续 3，贪心全对即学到循环；与 05-03 同前缀可比。

In [ ]:
gpt.eval(); mamba.eval()
import torch
prefixes = [torch.tensor([X_test[i][:3]], dtype=torch.long) for i in range(3)]
true_cont = [X_test[i][3:6] for i in range(3)]
with torch.no_grad():
    gpt_cont = [gpt.generate(p, max_new=3, temperature=0).tolist()[0][3:] for p in prefixes]
    mamba_cont = [mamba.generate(torch.tensor([X_test[i][:3]], dtype=torch.long), max_new=3, temperature=0).tolist()[0][3:] for i in range(3)]
for i in range(3):
    print(f"例{i+1} prefix={X_test[i][:3]} true={true_cont[i]} GPT={gpt_cont[i]} Mamba={mamba_cont[i]}  G{'✓' if gpt_cont[i]==true_cont[i] else '✗'} M{'✓' if mamba_cont[i]==true_cont[i] else '✗'}")

# fig5：条带（三行：前缀灰 + GPT 蓝 + Mamba 绿，错红）
fig, axes = plt.subplots(3, 1, figsize=(8.5, 4.2))
for ax, pre, tru, g, m in zip(axes, [X_test[i][:3] for i in range(3)], true_cont, gpt_cont, mamba_cont):
    ax.set_xlim(0,6); ax.set_ylim(0,2.0); ax.axis("off")
    for j, tok in enumerate(pre):
        ax.add_patch(plt.Rectangle((j+0.08, 1.05), 0.84, 0.55, facecolor="#D5D8DC", edgecolor="#555", linewidth=0.7))
        ax.text(j+0.5, 1.32, str(tok), ha="center", va="center")
    for j, (tok, gt, mt) in enumerate(zip(g, tru, m)):
        ok_g = (tok==gt); ok_m = (mt==gt)
        ax.add_patch(plt.Rectangle((3+j+0.08, 1.05), 0.84, 0.28, facecolor="#D6EAF8" if ok_g else "#FADBD8", edgecolor="#2E86C1" if ok_g else "#C0392B", linewidth=0.7))
        ax.text(3+j+0.5, 1.19, f"G:{tok}", ha="center", va="center", fontsize=8, color="#2E86C1" if ok_g else "#C0392B")
        ax.add_patch(plt.Rectangle((3+j+0.08, 0.48), 0.84, 0.28, facecolor="#D5F5E3" if ok_m else "#FADBD8", edgecolor="#1E8449" if ok_m else "#C0392B", linewidth=0.7))
        ax.text(3+j+0.5, 0.62, f"M:{mt}", ha="center", va="center", fontsize=8, color="#1E8449" if ok_m else "#C0392B")
    ax.set_title(f"prefix {pre} true {tru}  G {g} {'✓' if g==tru else '✗'}  M {m} {'✓' if m==tru else '✗'}", fontsize=9, loc="left")
plt.suptitle("同前缀续写 3 例：GPT vs Mamba（线性亦学到循环）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig5_gen.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. 总结与下一步

**本项目收获**

1. 同 LM 同预算次二次 vs 线性同台：循环计数 800/200 S=8 双 next-acc 高分（GPT 与 Mamba），线性亦学有序
2. 理论+实测：FLOPs/耗时随 S 二次 vs 线性曲线，S>32 差距拉开
3. Mamba 极简实现闭环：selective dt/B/C + 递推 h，无 S×S 矩阵
4. 拓展定位：本章与 04-04 共用实现，此处聚焦与 Transformer 对照；后续 08 家族再看 FlashAttention/KV-Cache 对二次的工程缓解

**下一步**：05 家族主干 01-04 + 拓展 05 全部闭环，后续 `06_Transformer_Vision_Multimodal`（视觉/多模态 Transformer）。